# Drift Batch Tracker

Use this notebook to keep the OpenDrift workflow notebook-based.

It does four things:
- inventories every dated scene in `Drift/<YYYY-MM-DD>`
- flags which SAR scenes already have drift outputs and which still need to run
- confirms the matching MARIDA seed shapefile for each scene
- builds Planet PSScene quick-search requests for the current Drift scenes

The existing broader Planet inventory is still available at `../SAR_inventory_planet.docx`, but this notebook gives you a Drift-specific view.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import re
from datetime import datetime, timedelta, timezone
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import rasterio
from pyproj import CRS, Transformer
from rasterio.warp import transform_bounds

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)


NOTEBOOK_DIR = Path.cwd()
DRIFT_ROOT = NOTEBOOK_DIR if NOTEBOOK_DIR.name == "Drift" else NOTEBOOK_DIR / "Drift"
if not DRIFT_ROOT.exists():
    DRIFT_ROOT = Path("/mnt/d/Masters/Drift")

MASTERS_ROOT = DRIFT_ROOT.parent
SHP_ROOT = MASTERS_ROOT / "MARIDA" / "MARIDA" / "shapefiles"
PLANET_DOCX = MASTERS_ROOT / "SAR_inventory_planet.docx"
PLANET_SCRIPT = MASTERS_ROOT / "DataBuilding.py"
BATCH_SCRIPT = DRIFT_ROOT / "run_opendrift_batch.py"
PLANET_WINDOW_HOURS = 12


def load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


def iter_date_dirs(drift_root: Path = DRIFT_ROOT) -> list[Path]:
    return sorted(path for path in drift_root.iterdir() if path.is_dir() and re.match(r"\d{4}-\d{2}-\d{2}$", path.name))


def get_optical_ref(date_dir: Path) -> Path | None:
    return next((date_dir / "optical").glob("S2_*_B02.tif"), None)


def parse_tile_date_from_optical(ref_tif: Path) -> tuple[str, datetime.date]:
    match = re.match(r"S2_([A-Z0-9]{5})_(\d{4}-\d{2}-\d{2})_B02\.tif$", ref_tif.name)
    if not match:
        raise ValueError(f"Unexpected optical name: {ref_tif.name}")
    tile = match.group(1)
    date_obj = pd.to_datetime(match.group(2)).date()
    return tile, date_obj


def candidate_shp_names(date_obj, tile: str) -> list[str]:
    yy = f"{date_obj.year % 100:02d}"
    day_z, day_n = f"{date_obj.day:02d}", str(date_obj.day)
    month_z, month_n = f"{date_obj.month:02d}", str(date_obj.month)
    stems = [
        f"S2_{day_n}-{month_n}-{yy}_{tile}",
        f"S2_{day_z}-{month_n}-{yy}_{tile}",
        f"S2_{day_n}-{month_z}-{yy}_{tile}",
        f"S2_{day_z}-{month_z}-{yy}_{tile}",
    ]
    return [stem + ".shp" for stem in stems]


def find_matching_shapefile(shp_root: Path, tile: str, date_obj) -> Path | None:
    for name in candidate_shp_names(date_obj, tile):
        path = shp_root / name
        if path.exists():
            return path

    pattern = re.compile(r"S2_(\d{1,2})-(\d{1,2})-(\d{2})_([A-Z0-9]{5})\.shp$")
    for path in shp_root.glob("S2_*-*-*_*.shp"):
        match = pattern.match(path.name)
        if not match:
            continue
        day, month, yy2, this_tile = match.groups()
        if this_tile != tile:
            continue
        try:
            this_date = pd.to_datetime(f"20{yy2}-{int(month):02d}-{int(day):02d}").date()
        except Exception:
            continue
        if this_date == date_obj:
            return path
    return None

print(f"Drift root: {DRIFT_ROOT}")
print(f"Shapefile root: {SHP_ROOT} (exists={SHP_ROOT.exists()})")
print(f"Batch script: {BATCH_SCRIPT}")
print(f"Planet generator: {PLANET_SCRIPT} (exists={PLANET_SCRIPT.exists()})")
print(f"Planet inventory doc: {PLANET_DOCX} (exists={PLANET_DOCX.exists()})")


In [ ]:
def scene_id_from_sar_dir(sar_dir: Path) -> str:
    manifests = sorted(sar_dir.glob("*_slc_manifest.json"))
    if manifests:
        try:
            payload = json.loads(manifests[0].read_text())
            scene_id = payload.get("scene_id")
            if scene_id:
                return scene_id
        except Exception:
            pass
        return manifests[0].stem.replace("_slc_manifest", "")

    for pattern, suffix in (("*_vv.tif", "_vv"), ("*_vh.tif", "_vh")):
        tif = next(sar_dir.glob(pattern), None)
        if tif is not None:
            return tif.stem.replace(suffix, "")

    return sar_dir.name


def scene_timestamp_utc(sar_dir: Path) -> datetime | None:
    scene_id = scene_id_from_sar_dir(sar_dir)
    match = re.search(r"_(\d{8}T\d{6})$", scene_id)
    if not match:
        match = re.search(r"_(\d{8}T\d{6})_", scene_id)
    if not match:
        return None
    return datetime.strptime(match.group(1), "%Y%m%dT%H%M%S").replace(tzinfo=timezone.utc)


def safe_float(value, ndp: int = 6):
    try:
        return round(float(value), ndp)
    except Exception:
        return None


def centroid_lonlat_from_raster(path: Path) -> tuple[float | None, float | None]:
    with rasterio.open(path) as ds:
        bounds = ds.bounds
        if ds.crs is None:
            lon = (bounds.left + bounds.right) / 2.0
            lat = (bounds.top + bounds.bottom) / 2.0
            return safe_float(lon), safe_float(lat)

        crs = CRS.from_user_input(ds.crs)
        try:
            tb = transform_bounds(crs, "EPSG:4326", bounds.left, bounds.bottom, bounds.right, bounds.top, densify_pts=21)
            lon = (tb[0] + tb[2]) / 2.0
            lat = (tb[1] + tb[3]) / 2.0
            return safe_float(lon), safe_float(lat)
        except Exception:
            cx, cy = ds.transform * (ds.width / 2.0, ds.height / 2.0)
            transformer = Transformer.from_crs(crs, CRS.from_epsg(4326), always_xy=True)
            lon, lat = transformer.transform(cx, cy)
            return safe_float(lon), safe_float(lat)


def planet_curl_for_point_psscene(lon: float, lat: float, center_dt: datetime | None, window_hours: int = PLANET_WINDOW_HOURS) -> str:
    if center_dt is None:
        gte = "1970-01-01T00:00:00Z"
        lte = "2100-01-01T00:00:00Z"
    else:
        start = (center_dt - timedelta(hours=window_hours)).astimezone(timezone.utc)
        end = (center_dt + timedelta(hours=window_hours)).astimezone(timezone.utc)
        gte = start.strftime("%Y-%m-%dT%H:%M:%SZ")
        lte = end.strftime("%Y-%m-%dT%H:%M:%SZ")

    payload = {
        "item_types": ["PSScene"],
        "filter": {
            "type": "AndFilter",
            "config": [
                {
                    "type": "GeometryFilter",
                    "field_name": "geometry",
                    "config": {"type": "Point", "coordinates": [float(lon), float(lat)]},
                },
                {
                    "type": "DateRangeFilter",
                    "field_name": "acquired",
                    "config": {"gte": gte, "lte": lte},
                },
            ],
        },
        "sort": [{"field_name": "acquired", "direction": "asc"}],
        "limit": 1,
    }
    payload_json = json.dumps(payload, separators=(",", ":"))
    return (
        'curl -u "$PL_API_KEY:" -X POST "https://api.planet.com/data/v1/quick-search" '
        '-H "Content-Type: application/json" '
        f"-d '{payload_json}'"
    )


def scan_drift_scene_status(drift_root: Path = DRIFT_ROOT, shp_root: Path = SHP_ROOT) -> pd.DataFrame:
    rows = []
    for date_dir in iter_date_dirs(drift_root):
        ref_tif = get_optical_ref(date_dir)
        has_optical = ref_tif is not None
        has_bio_s2 = (date_dir / "bio_s2").exists()
        tile = None
        date_iso = date_dir.name
        shp_path = None

        if has_optical:
            tile, date_obj = parse_tile_date_from_optical(ref_tif)
            date_iso = str(date_obj)
            shp_path = find_matching_shapefile(shp_root, tile, date_obj)

        sar_dirs = sorted(path for path in date_dir.iterdir() if path.is_dir() and path.name.startswith("SAR_"))
        for sar_dir in sar_dirs:
            scene_id = scene_id_from_sar_dir(sar_dir)
            vv_tif = next(sar_dir.glob("*_vv.tif"), None)
            vh_tif = next(sar_dir.glob("*_vh.tif"), None)
            centroid_src = vv_tif or vh_tif
            lon, lat = centroid_lonlat_from_raster(centroid_src) if centroid_src else (None, None)
            seed_count_id1 = None
            if shp_path is not None:
                try:
                    shp_gdf = gpd.read_file(shp_path)
                    if "id" in shp_gdf.columns:
                        seed_count_id1 = int((shp_gdf["id"] == 1).sum())
                    else:
                        seed_count_id1 = len(shp_gdf)
                except Exception:
                    seed_count_id1 = None
            has_id1_seeds = seed_count_id1 is not None and seed_count_id1 > 0
            required_inputs = all(
                [
                    has_optical,
                    has_bio_s2,
                    (sar_dir / "bio").exists(),
                    (sar_dir / "SLC").exists(),
                    shp_path is not None,
                    has_id1_seeds,
                ]
            )
            has_predicted = (sar_dir / "predicted_points_plast.shp").exists()
            has_boxes = (sar_dir / "search_boxes_1km_plast.shp").exists()
            has_forcing = (sar_dir / "forcing_plast.nc").exists()
            has_quicklook = (sar_dir / "quicklook.png").exists()
            if has_predicted and has_boxes:
                status = "done"
            elif required_inputs:
                status = "pending"
            else:
                status = "blocked"

            rows.append(
                {
                    "date": date_iso,
                    "date_dir": date_dir,
                    "tile": tile,
                    "sar_dir_name": sar_dir.name,
                    "sar_dir": sar_dir,
                    "scene_id": scene_id,
                    "scene_time_utc": scene_timestamp_utc(sar_dir),
                    "has_optical": has_optical,
                    "has_bio_s2": has_bio_s2,
                    "has_sar_bio": (sar_dir / "bio").exists(),
                    "has_slc_dir": (sar_dir / "SLC").exists(),
                    "has_slc_manifest": any(sar_dir.glob("*_slc_manifest.json")),
                    "has_vv": vv_tif is not None,
                    "has_vh": vh_tif is not None,
                    "has_shapefile": shp_path is not None,
                    "has_id1_seeds": has_id1_seeds,
                    "seed_count_id1": seed_count_id1,
                    "shapefile": shp_path,
                    "has_forcing": has_forcing,
                    "has_predicted_points": has_predicted,
                    "has_search_boxes": has_boxes,
                    "has_quicklook": has_quicklook,
                    "status": status,
                    "needs_drift": status == "pending",
                    "centroid_lon": lon,
                    "centroid_lat": lat,
                }
            )

    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame
    return frame.sort_values(["date", "sar_dir_name"]).reset_index(drop=True)


scene_status = scan_drift_scene_status()
summary_cols = [
    "date",
    "tile",
    "sar_dir_name",
    "scene_id",
    "status",
    "seed_count_id1",
    "has_shapefile",
    "has_forcing",
    "has_predicted_points",
    "has_search_boxes",
    "has_quicklook",
]

display(scene_status[summary_cols])
pending_scenes = scene_status.loc[scene_status["needs_drift"], summary_cols].reset_index(drop=True)
print(f"Pending scenes: {len(pending_scenes)}")
display(pending_scenes)


In [ ]:
def build_planet_inventory(scene_frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for row in scene_frame.itertuples(index=False):
        if row.centroid_lon is None or row.centroid_lat is None:
            continue
        rows.append(
            {
                "date": row.date,
                "sar_dir_name": row.sar_dir_name,
                "scene_id": row.scene_id,
                "scene_time_utc": row.scene_time_utc,
                "status": row.status,
                "needs_drift": row.needs_drift,
                "centroid_lon": row.centroid_lon,
                "centroid_lat": row.centroid_lat,
                "planet_quick_search_curl": planet_curl_for_point_psscene(
                    row.centroid_lon,
                    row.centroid_lat,
                    row.scene_time_utc,
                ),
            }
        )

    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame
    return frame.sort_values(["date", "sar_dir_name"]).reset_index(drop=True)


planet_inventory = build_planet_inventory(scene_status)
planet_csv = DRIFT_ROOT / "drift_planet_inventory.csv"
planet_inventory.to_csv(planet_csv, index=False)

print(f"Drift-specific Planet inventory saved to: {planet_csv}")
print(f"Existing broader inventory doc: {PLANET_DOCX} (exists={PLANET_DOCX.exists()})")
display(
    planet_inventory[
        [
            "date",
            "sar_dir_name",
            "scene_id",
            "scene_time_utc",
            "status",
            "needs_drift",
            "centroid_lon",
            "centroid_lat",
            "planet_quick_search_curl",
        ]
    ]
)


In [ ]:
def _stretch(values, percentiles: tuple[int, int] = (2, 98)):
    lo, hi = pd.Series(values.ravel()).quantile([percentiles[0] / 100.0, percentiles[1] / 100.0])
    return values.clip(lo, hi)


def save_quicklook(date_dir: Path, sar_dir: Path, shp_path: Path, out_png: Path | None = None) -> Path:
    with rasterio.open(get_optical_ref(date_dir)) as ds:
        ref = ds.read(1).astype("float32")
        bounds = ds.bounds
        extent = [float(bounds.left), float(bounds.right), float(bounds.bottom), float(bounds.top)]
        crs = ds.crs

    seeds = gpd.read_file(shp_path)
    if "id" in seeds.columns:
        seeds = seeds[seeds["id"] == 1].copy()
    seeds = seeds.to_crs(crs)

    preds = gpd.read_file(sar_dir / "predicted_points_plast.shp").to_crs(crs)
    boxes = gpd.read_file(sar_dir / "search_boxes_1km_plast.shp").to_crs(crs)

    if out_png is None:
        out_png = sar_dir / "quicklook.png"

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(
        _stretch(ref),
        extent=extent,
        origin="upper",
        cmap="gray",
    )
    if not seeds.empty:
        seeds.plot(ax=ax, markersize=10, color="deepskyblue", label="Seeds (id=1)")
    if not preds.empty:
        preds.plot(ax=ax, markersize=10, color="crimson", label="Predicted points")
    if not boxes.empty:
        boxes.boundary.plot(ax=ax, linewidth=1.0, color="orange", label="1 km boxes")
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_title(f"{date_dir.name} | {sar_dir.name}")
    ax.legend(loc="upper right")
    ax.set_axis_off()
    fig.savefig(out_png, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return out_png


def load_batch_runtime():
    return load_module("run_opendrift_batch_runtime", BATCH_SCRIPT)


def run_scene(date_dir: Path, sar_dir: Path, shp_path: Path, include_existing: bool = False, save_preview: bool = True) -> str:
    out_shp = sar_dir / "predicted_points_plast.shp"
    if out_shp.exists() and not include_existing:
        if save_preview and not (sar_dir / "quicklook.png").exists() and (sar_dir / "search_boxes_1km_plast.shp").exists():
            save_quicklook(date_dir, sar_dir, shp_path)
        return "skipped"

    batch_runtime = load_batch_runtime()
    batch_runtime.run_plast(date_dir, sar_dir, shp_path)
    if save_preview:
        save_quicklook(date_dir, sar_dir, shp_path)
    return "done"


def run_scenes(
    dates: list[str] | None = None,
    pending_only: bool = True,
    include_existing: bool = False,
    save_previews: bool = True,
) -> pd.DataFrame:
    current = scan_drift_scene_status()
    selected = current.copy()
    if pending_only:
        selected = selected[selected["needs_drift"]]
    if dates:
        selected = selected[selected["date"].isin(dates)]
    selected = selected[selected["has_shapefile"]]

    if selected.empty:
        print("No scenes selected.")
        return current

    results = []
    for row in selected.itertuples(index=False):
        print(f"Running {row.date} / {row.sar_dir_name} ...")
        try:
            outcome = run_scene(
                Path(row.date_dir),
                Path(row.sar_dir),
                Path(row.shapefile),
                include_existing=include_existing,
                save_preview=save_previews,
            )
            results.append({"date": row.date, "sar_dir_name": row.sar_dir_name, "result": outcome})
        except Exception as exc:
            results.append({"date": row.date, "sar_dir_name": row.sar_dir_name, "result": f"failed: {exc}"})

    results_df = pd.DataFrame(results)
    display(results_df)
    refreshed = scan_drift_scene_status()
    display(refreshed[summary_cols])
    return refreshed


In [ ]:
# Example usage:
# updated_status = run_scenes(dates=["2020-09-15", "2021-01-23"], pending_only=False, include_existing=False, save_previews=True)
# display(updated_status[summary_cols])
